# BlendColab / GodotBakeCloud — Distributed Free Cloud GPU Farm

Render **Blender Cycles** animations (supporting **Blender 3.6 LTS through 5.3** with `OptiX` on T4 / `CUDA` on P100) or bake **Godot 4 `LightmapGI`** scenes headlessly using Google Drive as a shared distributed lock queue.

### Multi-GPU Scaling Tip
- Open **2 to 4 browser tabs** (Google Colab + Kaggle `T4 x2`) pointed at the same `DRIVE_WORKSPACE` folder.
- Workers automatically claim unlocked frames/scenes, send heartbeats every 30s, recover orphaned locks if a tab disconnects, and push preview thumbnails to Discord.

In [ ]:
#@title 1. Mount Google Drive & Choose Blender Version
import json
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/DeviouslyCalamitous/BlendColab-GodotBakeCloud.git" #@param {type:"string"}
DRIVE_WORKSPACE = "BlendColab_Farm" #@param {type:"string"}
BLENDER_VERSION = "5.2 LTS (5.2.4)" #@param ["3.6 LTS (3.6.23)", "4.0 (4.0.2)", "4.1 (4.1.1)", "4.2 LTS (4.2.8)", "4.3 (4.3.2)", "4.4 (4.4.3)", "4.5 LTS (4.5.16)", "5.0 (5.0.1)", "5.1 (5.1.1)", "5.2 LTS (5.2.4)", "5.3 (5.3.0)"]
INSTALL_ENGINES = "blender" #@param ["blender", "both", "godot"]

# 1. Step back to the root directory before removing any stale clone
base_dir = Path("/content") if Path("/content").exists() else Path("/kaggle/working")
os.chdir(base_dir)

Path("/tmp/cloud_farm_paths.json").unlink(missing_ok=True)
repo_dir = base_dir / "BlendColab-GodotBakeCloud"
if repo_dir.exists():
    shutil.rmtree(repo_dir)

# 2. Mount Google Drive
if Path("/content").exists():
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

# 3. Fresh clone directly from GitHub
print(f"Cloning fresh copy from {REPO_URL}...")
!git clone --depth 1 "{REPO_URL}" "{repo_dir}"
assert repo_dir.exists(), "Git clone failed — check the error message printed above."

os.chdir(repo_dir)

# Normalize any accidental non-breaking spaces from web copy-paste
for py_file in Path("scripts").glob("*.py"):
    raw = py_file.read_text(encoding="utf-8")
    if "\u00a0" in raw:
        py_file.write_text(raw.replace("\u00a0", " "), encoding="utf-8")

# 4. Download & cache the selected Blender version
!python3 scripts/env_bootstrap.py --engine "{INSTALL_ENGINES}" --blender-version "{BLENDER_VERSION}" --workspace-name "{DRIVE_WORKSPACE}" --repo-dir "{repo_dir}"

assert Path("/tmp/cloud_farm_paths.json").exists(), "Setup failed — check the error message printed above."

with open("/tmp/cloud_farm_paths.json", "r", encoding="utf-8") as f:
    PATHS = json.load(f)

print("\nDetected GPU Hardware:")
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv,noheader
print(f"\nLoaded Blender Version: {PATHS.get('blender_version')} -> {PATHS.get('blender_bin')}")
print(f"Shared Drive Workspace Ready: {PATHS['workspace']}")
print("Upload your .blend file into:", f"{PATHS['workspace']}/projects/")

In [ ]:
#@title 2. Configure & Start Distributed Render / Bake Worker
import json
import shlex
from pathlib import Path

assert Path("/tmp/cloud_farm_paths.json").exists(), "Please run Cell 1 first to connect Google Drive and load Blender!"

with open("/tmp/cloud_farm_paths.json", "r", encoding="utf-8") as f:
    PATHS = json.load(f)

#@markdown ### Engine & Project Selection
ENGINE = "blender" #@param ["blender", "godot"]
PROJECT_SUBPATH = "projects/scene.blend" #@param {type:"string"}

#@markdown ### Blender Cycles Settings (Ignored in Godot Mode)
START_FRAME = 1 #@param {type:"integer"}
END_FRAME = 120 #@param {type:"integer"}
SAMPLES_OVERRIDE = 0 #@param {type:"integer"}
ASSEMBLE_MP4 = True #@param {type:"boolean"}
FPS = 24 #@param {type:"integer"}
POOL_MULTI_GPU = False #@param {type:"boolean"}

#@markdown ### Godot 4 LightmapGI Settings (Ignored in Blender Mode)
GODOT_SCENES = "res://main.tscn" #@param {type:"string"}

#@markdown ### Telemetry & Quota Protection
DISCORD_WEBHOOK_URL = "" #@param {type:"string"}
NOTIFY_EVERY_N_TASKS = 5 #@param {type:"integer"}
AUTO_DISCONNECT_WHEN_DONE = False #@param {type:"boolean"}

workspace = Path(PATHS["workspace"])
project_path = workspace / PROJECT_SUBPATH
assert project_path.exists(), f"Project not found on Drive: {project_path}"

cmd = [
    "python3", "-u", "scripts/render_worker.py",
    "--engine", ENGINE,
    "--workspace", str(workspace),
    "--notify-every", str(max(1, NOTIFY_EVERY_N_TASKS)),
]

if DISCORD_WEBHOOK_URL.strip():
    cmd.extend(["--discord-webhook", DISCORD_WEBHOOK_URL.strip()])
if AUTO_DISCONNECT_WHEN_DONE:
    cmd.append("--auto-disconnect")
if POOL_MULTI_GPU:
    cmd.append("--pool-gpus")

if ENGINE == "blender":
    cmd.extend([
        "--blender-bin", PATHS["blender_bin"],
        "--blend-file", str(project_path),
        "--start-frame", str(START_FRAME),
        "--end-frame", str(END_FRAME),
        "--fps", str(FPS),
    ])
    if SAMPLES_OVERRIDE > 0:
        cmd.extend(["--samples", str(SAMPLES_OVERRIDE)])
    if ASSEMBLE_MP4:
        cmd.append("--assemble-video")
else:
    from scripts.env_bootstrap import inject_godot_addon
    inject_godot_addon(Path.cwd(), project_path)
    cmd.extend([
        "--godot-bin", PATHS["godot_bin"],
        "--godot-project", str(project_path),
        "--godot-scenes", GODOT_SCENES,
    ])

cmd_str = shlex.join(cmd)
print("Launching worker:\n ", cmd_str, "\n")
!{cmd_str}

In [ ]:
#@title 3. Queue Inspector & Lock Maintenance Utility
import json
import time
from pathlib import Path

assert Path("/tmp/cloud_farm_paths.json").exists(), "Please run Cell 1 first to connect Google Drive!"

with open("/tmp/cloud_farm_paths.json", "r", encoding="utf-8") as f:
    PATHS = json.load(f)

FORCE_CLEAR_ALL_LOCKS = False #@param {type:"boolean"}
RESET_DONE_MARKERS = False #@param {type:"boolean"}

queue_dir = Path(PATHS["workspace"]) / "render_queue"
done_files = sorted(queue_dir.glob("*.done"))
lock_files = sorted(queue_dir.glob("*.lock"))

print(f"Completed Tasks : {len(done_files)}")
print(f"Active Locks    : {len(lock_files)}")

for lf in lock_files:
    try:
        info = json.loads(lf.read_text(encoding="utf-8"))
        age = time.time() - info.get("heartbeat_at", 0)
        print(f"  - {lf.name}: worker={info.get('worker_id')} | gpu={info.get('gpu_name')} | heartbeat_age={age:.0f}s")
    except Exception:
        print(f"  - {lf.name}: unreadable lockfile")

if FORCE_CLEAR_ALL_LOCKS:
    for lf in lock_files:
        lf.unlink(missing_ok=True)
    print("Cleared all .lock files.")

if RESET_DONE_MARKERS:
    for df in done_files:
        df.unlink(missing_ok=True)
    print("Cleared all .done markers (queue reset for full re-render).")